# Zadanie 3: optymalizacja dyskretna

Termin realizacji: 20 kwietnia 2026

Zadanie do oddania przez MS Teams. Do oddania: kod oraz krótkie sprawozdanie w PDF (można na przykład przy użyciu `quarto render notebook.ipynb --to pdf`).

## Na 3.0

Do realizacji:

1. Zaimplementuj dyskretny problem plecakowy z trzema plecakami w MiniZinc na podstawie przykładu (plik `minizinc.ipynb`). Spróbuj rozwiązać problem dla 10 zestawów parametrów o różnych wielkościach tak, aby rozwiązanie największego problemu trwało powyżej 5 sekund. Zanotuj w każdym przypadku liczbę wszystkich przedmiotów, pojemności plecaków, liczbę wybranych przedmiotów i sumaryczną wartość przedmiotów w każdym plecaku osobno.
2. Losowanie przedmiotów w każdym przypadku powinno być tak ustawione, aby optymalne rozwiązanie wymagało wzięcia przynajmniej dwóch przedmiotów do każdego plecaka oraz zostawienia przynajmniej dwóch przedmiotów poza plecakami. W raporcie należy umieścić informację w jaki sposób zostało to zweryfikowane.
3. Zmodyfikuj metodę z notatnika `tabu_search.ipynb` tak aby rozwiązywała opisywany problem plecakowy. Porównaj na tych samych problemach czy Minizinc i Tabu search zwracają równie dobre rozwiazania, oraz wypisz jakie to są rozwiązania. Wykonaj eksperymenty z trzema różnymi długościami listy zakazów (1, 2, 5).

## Na 4.0

Do realizacji:

1. Punkty z zadania na 3.0.
2. Rozszerz możliwe ruchy w tabu search o przeniesienie przedmiotu z jednego plecaka do drugiego. Zapisz rozważ czy to poprawia działanie metody (czy znalezione jest lepsze, takie samo czy gorsze rozwiązanie? czy rozwiązanie jest znajdowane szybciej czy wolniej?). Dla każdego z 10 zestawów parametrów problemu plecakowego wykonaj ocenę przez uśrednienie dla 10 różnych losowych przypadków.
3. Podsumuj dane w formie tabelki z czterema kolumnami (Minizinc, tabu search z listą o długości 1, 2, i 5) oraz 10 wierszami (po jednym dla zestawu parametrów problemu), a w komórkach umieść średnią wartość wartości przedmiotów oraz średni czas potrzebny do uzyskania rozwiązania.

## Na 5.0

Do realizacji:

1. Punkty z zadania na 4.0.
2. Zaimplementuj samodzielnie algorytm symulowanego wyżarzania z podobnym interfejsem co tabu search. Porównaj jego działanie do rozważanych wcześniej rozwiązań dla trzech różnych schematów chłodzenia. Dobierz liczbę iteracji tak, aby czas działania był porównywalny do tabu search.


In [2]:
import Pkg
Pkg.add("Distributions")

    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
   Installed Rmath ─────────────────── v0.9.0
   Installed Rmath_jll ─────────────── v0.5.1+0
   Installed StatsFuns ─────────────── v1.5.2
   Installed PDMats ────────────────── v0.11.37
   Installed HypergeometricFunctions ─ v0.3.28
   Installed QuadGK ────────────────── v2.11.3
   Installed Distributions ─────────── v0.25.124
  Installing 1 artifacts
   Installed artifact Rmath                    121.9 KiB
    Updating `~/.julia/environments/v1.12/Project.toml`
  [31c24e10] + Distributions v0.25.124
    Updating `~/.julia/environments/v1.12/Manifest.toml`
  [31c24e10] + Distributions v0.25.124
  [34004b35] + HypergeometricFunctions v0.3.28
  [90014a1f] + PDMats v0.11.37
  [1fd47b50] + QuadGK v2.11.3
  [79098fc4] + Rmath v0.9.0
  [4c63d2b9] + StatsFuns v1.5.2
  [f50d1b31] + Rmath_jll v0.5.1+0
  [4607b0f0] + SuiteSparse
Precompiling packages...
    917.6 ms  ✓ Rmath_jll
   1258.3 ms  ✓ PDMats


In [3]:
using Distributions

function make_dzn(n::Int, capacity::Int)
    profits = rand(DiscreteUniform(10, 1000), n)
    weights = rand(DiscreteUniform(10, 100), n)
    content = """
    ITEM = _(1..$n);
    capacity = $capacity;
    profits = $profits;
    weights = $weights;
    """
    file = open("knapsack_generated.dzn", "w+")
    write(file, content)
    close(file)
end
make_dzn(100, 200)

In [4]:
function make_dzn_trip(n::Int, m::Int,cap_min::Int,cap_max::Int)
    profits = rand(DiscreteUniform(10, 1000), n)
    weights = rand(DiscreteUniform(10, 100), n)
    capacities = rand(DiscreteUniform(cap_min, cap_max), m)

    content = """
    n = $n;
    m = $m;
    capacities = $capacities;
    profits = $profits;
    weights = $weights;
    """

    file = open("knapsack_triple_generated.dzn", "w+")
    write(file, content)
    close(file)
end

make_dzn_trip (generic function with 1 method)

In [ ]:
make_dzn_trip(150,3,400,700)

Przedmioty w plecakach:
Plecak 1: [3, 23, 31, 42, 51, 73, 88, 89, 101, 105, 112, 140, 146, 147] | Wartość: 9859
Plecak 2: [5, 16, 24, 43, 69, 91, 96, 104, 107, 111, 116, 120, 123, 130, 132, 135, 136, 144] | Wartość: 12689
Plecak 3: [17, 22, 25, 36, 62, 67, 77, 84, 90, 97, 110, 139, 145, 148] | Wartość: 9106

Przedmioty w plecakach:
Plecak 1: [3, 17, 23, 31, 42, 51, 88, 89, 101, 105, 112, 132, 140, 146, 147] | Wartość: 11080
Plecak 2: [16, 24, 25, 43, 69, 73, 76, 91, 96, 104, 107, 111, 116, 120, 123, 130, 135, 136, 144] | Wartość: 12561
Plecak 3: [5, 22, 36, 62, 77, 84, 86, 90, 97, 110, 115, 139, 145, 148] | Wartość: 9192

Przedmioty w plecakach:
Plecak 1: [3, 17, 23, 36, 40, 42, 62, 66, 77, 84, 90, 96, 97, 107, 115, 120, 139, 145] | Wartość: 11795
Plecak 2: [5, 16, 22, 31, 51, 69, 88, 89, 110, 111, 123, 132, 140, 146, 147] | Wartość: 11126
Plecak 3: [24, 25, 43, 67, 73, 91, 101, 104, 105, 116, 130, 135, 136, 144, 148] | Wartość: 10041

==========
Finished in 5s 403msec.

In [8]:
make_dzn_trip(100,3,5000,10000)

Przedmioty w plecakach:
Plecak 1: [7, 8, 13, 14, 15, 16, 18, 20, 24, 26, 27, 28, 30, 31, 32, 35, 36, 38, 39, 40, 41, 43, 45, 47, 48, 49, 50, 51, 53, 55, 58, 60, 61, 62, 65, 66, 68, 69, 72, 76, 80, 82, 84, 85, 87, 90, 91, 92, 94, 97] | Wartość: 29448
Plecak 2: [9, 11, 12, 21, 23, 29, 33, 46, 56, 57, 59, 63, 67, 88, 93, 96, 100] | Wartość: 7557
Plecak 3: [1, 2, 3, 4, 5, 6, 10, 17, 19, 22, 25, 34, 37, 42, 44, 52, 54, 64, 70, 71, 73, 74, 75, 77, 78, 79, 81, 83, 86, 89, 95, 98, 99] | Wartość: 16499

==========
Finished in 116msec.

In [9]:
make_dzn_trip(100,3,100,300)

Przedmioty w plecakach:
Plecak 1: [20, 23, 35, 42, 52, 68, 80, 91, 93, 96] | Wartość: 7409
Plecak 2: [1, 2, 24, 31, 33, 76, 78, 87] | Wartość: 6394
Plecak 3: [7, 9, 12, 34, 37, 44, 59, 84, 90, 92, 98] | Wartość: 7713

Przedmioty w plecakach:
Plecak 1: [1, 9, 23, 35, 42, 44, 80, 84, 91, 93, 96, 98] | Wartość: 8885
Plecak 2: [20, 24, 31, 33, 52, 76, 78, 87] | Wartość: 6079
Plecak 3: [2, 7, 12, 34, 37, 59, 72, 83, 90, 92] | Wartość: 7085

Przedmioty w plecakach:
Plecak 1: [1, 9, 23, 52, 76, 87, 90, 93, 98] | Wartość: 7329
Plecak 2: [2, 12, 20, 31, 37, 44, 59, 68, 80, 84, 91] | Wartość: 7707
Plecak 3: [5, 7, 24, 33, 34, 35, 42, 78, 92, 96] | Wartość: 7083

==========
Finished in 3s 479msec.

In [10]:
make_dzn_trip(100,3,10,20)

=====UNSATISFIABLE=====
Finished in 222msec.

In [11]:
make_dzn_trip(100,3,500,2000)

Przedmioty w plecakach:
Plecak 1: [2, 3, 4, 5, 8, 16, 19, 20, 21, 23, 26, 30, 35, 37, 40, 43, 45, 48, 50, 52, 58, 62, 64, 68, 71, 77, 81, 83, 84, 86, 88, 89, 91, 93, 94, 99] | Wartość: 19295
Plecak 2: [1, 6, 10, 14, 17, 18, 22, 25, 31, 46, 61, 69, 70, 72, 73, 75, 78, 87, 95] | Wartość: 10153
Plecak 3: [7, 9, 12, 44, 47, 55, 57, 59, 60, 65, 66, 67, 82, 98] | Wartość: 9050

Przedmioty w plecakach:
Plecak 1: [2, 3, 4, 5, 8, 16, 19, 20, 21, 23, 26, 30, 35, 37, 40, 43, 45, 48, 50, 52, 58, 62, 64, 68, 71, 77, 81, 83, 84, 86, 88, 89, 91, 93, 94, 99] | Wartość: 19295
Plecak 2: [1, 6, 10, 14, 17, 18, 22, 25, 31, 46, 61, 69, 72, 73, 75, 78, 87, 90, 95] | Wartość: 10164
Plecak 3: [7, 9, 12, 44, 47, 55, 57, 59, 60, 65, 66, 67, 79, 82] | Wartość: 9661

Przedmioty w plecakach:
Plecak 1: [2, 4, 5, 8, 16, 19, 20, 21, 23, 26, 30, 35, 37, 40, 43, 45, 48, 50, 52, 58, 62, 64, 68, 69, 71, 77, 81, 83, 84, 86, 88, 89, 91, 93, 94, 99] | Wartość: 19090
Plecak 2: [1, 3, 10, 14, 15, 17, 18, 22, 25, 31, 46, 61, 70, 72, 73, 75, 76, 78, 87, 90, 95] | Wartość: 10960
Plecak 3: [7, 9, 12, 44, 47, 51, 55, 57, 59, 60, 65, 66, 67, 79, 82] | Wartość: 9800

Przedmioty w plecakach:
Plecak 1: [1, 2, 3, 5, 6, 20, 21, 23, 26, 30, 35, 37, 40, 43, 45, 48, 58, 60, 62, 64, 66, 67, 68, 71, 72, 77, 81, 82, 83, 84, 86, 89, 91, 93, 99] | Wartość: 18567
Plecak 2: [10, 14, 17, 18, 22, 25, 31, 50, 61, 65, 69, 70, 73, 75, 87, 88, 90, 94, 95] | Wartość: 11391
Plecak 3: [7, 8, 9, 12, 16, 19, 44, 47, 52, 55, 57, 59, 78, 79, 98] | Wartość: 10004

Przedmioty w plecakach:
Plecak 1: [2, 3, 5, 8, 10, 16, 19, 20, 21, 23, 26, 30, 35, 37, 40, 43, 44, 45, 48, 50, 62, 64, 68, 71, 73, 77, 81, 83, 84, 86, 88, 89, 91, 93, 94, 99] | Wartość: 19503
Plecak 2: [1, 6, 14, 18, 22, 25, 31, 46, 58, 59, 61, 69, 70, 72, 75, 78, 87, 90, 95] | Wartość: 10007
Plecak 3: [7, 9, 12, 17, 47, 52, 55, 57, 60, 65, 66, 67, 79, 82, 98] | Wartość: 10626

==========
Finished in 360msec.